In [1]:
from huggingface_hub import login
import os
from dotenv import load_dotenv
load_dotenv()

my_token = os.getenv("HF")

login(token=my_token)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer, AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

raw_text = ["This library is amazing!", "I am feeling very good today", "I am eating raw eggplant"]

inputs = tokenizer(raw_text, padding=True, truncation=True, return_tensors="pt")

outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8167.27it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.weight | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([3, 9, 768])


In [4]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

outputs = model(**inputs)

print(outputs.logits)


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 9033.27it/s]

tensor([[-4.3725,  4.7083],
        [-4.2794,  4.6810],
        [ 1.3332, -1.2696]], grad_fn=<AddmmBackward0>)


In [ ]:
import torch

logits = outputs.logits

probabilities = torch.nn.functional.softmax(logits, dim=-1)
for i, sentence in enumerate(raw_text):
    predicted_id = torch.argmax(logits[i]).item()

    label = model.config.id2label[predicted_id]
    
    confidence = probabilities[i][predicted_id].item() * 100
    
    print(f"Sentence: '{sentence}'")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2f}%\n")

1
Sentence: 'This library is amazing!'
Prediction: POSITIVE
Confidence: 99.99%

1
Sentence: 'I am feeling very good today'
Prediction: POSITIVE
Confidence: 99.99%

0
Sentence: 'I am eating raw eggplant'
Prediction: NEGATIVE
Confidence: 93.10%

